In [ ]:
from platform import python_version
print(python_version())

## Spatiotemporal structure of cell fate decisions in murine neural crest (forked)

[DOI: https://doi.org/10.1038/s41588-025-02352-6](https://www.science.org/doi/10.1126/science.aas9536?referrer=https%3A%2F%2Fwww.google.com%2F#sec-10)


![neural_crest_dev](../pictures/neural_crest_dev.jpeg)


And the published innervation result reads: RNA velocity and pseudotime analyses confirmed a fork-like transition from early SCPs towards two parallel trajectories of neuronal-chromaffin and glial cell lineages. The fork is neuronal-chromaffin versus glial — not chromaffin versus neuronal. I tested the wrong split. Re-testing against the actual claim:


### Data storage

???

Worth remembering the asymmetry: 

if you launch a kernel without the conda env active, the helper adds $R_HOME/bin (R-only, safe). If you launch with it active, conda's bin is already ahead and carries a python — harmless as long as .venv/bin precedes it, which check_r_stack doesn't verify. shutil.which("python") is the one-line check if a subprocess ever behaves oddly.

In [ ]:
# os.environ["R_HOME"] = "/home/flavio/miniforge3/envs/renv/lib/R"
# os.environ["R_LIBS_USER"] = "/home/flavio/miniforge3/envs/renv/lib/R/library"

In [ ]:
import os, sys, shutil
from pathlib import Path

SRC = next(p / "src" for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "libs").is_dir())
sys.path.insert(0, str(SRC))

from libs.paths import PROJ, H5AD, TABLES, FIGURES      # no chdir needed

#  R_HOME, LD_LIBRARY_PATH." LD_LIBRARY_PATH is not needed — your rpy2 was linked with an rpath, and setting it from inside Python has no effect anyway since the loader reads it at process start. 
# # The second requirement is R on PATH, for scFates' shutil.which("R") check. So: R_HOME + PATH.
# os.environ["R_HOME"] = "/home/flavio/miniforge3/envs/renv/lib/R"
# os.environ["R_LIBS_USER"] = "/home/flavio/miniforge3/envs/renv/lib/R/library"
from libs.run_scfates import preprocess, fit_curve      # sets R_HOME + PATH, then imports scFates
import rpy2.situation as s

print("python :", sys.executable)
print("R      :", shutil.which("R"), "| R_HOME:", os.environ.get("R_HOME"))
print("R in rpy2", s.get_r_home())
PROJ, H5AD, TABLES, FIGURES

In [ ]:
import importlib
_f = importlib.import_module("scFates.tools.test_association")
print("R stack:", all(not isinstance(getattr(_f, module), str)
                      for module in ("Rpy2", "R", "rstats", "rmgcv", "Formula")))

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/neural_crest")
ROOT_SRC = ROOT0 / "src"

root_figure = ROOT0 / "pictures"
root_results = ROOT0 / "results"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

ROOT0_DATA = create_dir(ROOT0, "data")
root_colab = create_dir(ROOT0_DATA, "colab")

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']
Worth remembering the asymmetry: if you launch a kernel without the conda env active, the helper adds $R_HOME/bin (R-only, safe). If you launch with it active, conda's bin is already ahead and carries a python — harmless as long as .venv/bin precedes it, which check_r_stack doesn't verify. shutil.which("python") is the one-line check if a subprocess ever behaves oddly.
case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']


### Scanpy Introduction

https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#

In [ ]:
# Core scverse libraries
from __future__ import annotations

# import anndata as ad

# Pooch is a Python library that can manage data by downloading files from a server (only when needed) and storing them locally in a data cache 
# import pooch
import scanpy as sc
import squidpy as sq
import importlib

# scanpy 1.12.4 | squidpy 1.8.3 | anndata 0.13.3.post0
# import anndata
import json

importlib.metadata.version("scanpy"), importlib.metadata.version("squidpy"), importlib.metadata.version("anndata")


In [ ]:
sc.set_figure_params(dpi=120, facecolor="white")

### Getting data

#### All single-cell RNA-seq datasets

- have been deposited in the GEO under accession code GSE129114. 

#### Processed data, code, supplementary materials, and interactive views of datasets 

- can be accessed on the authors’ website: http://pklab.med.harvard.edu/ruslan/neural.crest.html. 

anndata: https://anndata.scverse.org/en/stable/tutorials/notebooks/getting-started.html

In [ ]:
ROOT0_DATA

In [ ]:
root_h5 = create_dir(ROOT0_DATA, "h5ad")
os.listdir(root_h5)

In [ ]:
root_meta = create_dir(ROOT0_DATA, "meta")
root_results = create_dir(ROOT0_DATA, 'results')
os.listdir(root_results)

In [ ]:
import sys, os
from libs.assemble_h5ad import assemble

In [ ]:
root_bundles = create_dir(ROOT0_DATA, "bundles")
root_bundles_in = create_dir(root_bundles, "IN_sel")
os.listdir(root_bundles_in)

In [ ]:
adata = assemble(root_bundles_in, root_h5 / "innervation.h5ad")   # h5ad written to my workspace (grant is read-only)

In [ ]:
print("\nobs columns:")
for c in adata.obs.columns:
    s = adata.obs[c]
    nun = s.nunique(dropna=True)
    na = int(s.isna().sum())
    head = list(s.dropna().unique()[:6]) if nun <= 40 else f"{s.min()}..{s.max()}"
    print(f"  {c:32s} n={nun:4d} NA={na:5d}  {head}")

In [ ]:
ct = adata.obs["cell.types"].astype(str)

";".join(ct[:10]) + '...'

In [ ]:
adata.obs.columns

In [ ]:
adata.obs.head(3).T

In [ ]:
tab = (pd.crosstab(ct, adata.obs["clusters_subset"])
         .pipe(lambda d: d.loc[d.sum(1).sort_values(ascending=False).index]))

tab

In [ ]:
counts = ct.value_counts()
print("cell.types (n cells), and the clusters_subset id each maps to:")
for t, n in counts.items():
    ids = tab.columns[tab.loc[t] > 0].tolist()
    print(f"  {t:12s} {n:5d}  -> clusters_subset {ids}")
excl = ct.str.startswith("IN_excl")
print(f"\nIN_excl* cells: {int(excl.sum())} | remaining: {int((~excl).sum())} of {adata.n_obs}")
print("\nage_groups x excluded:\n", pd.crosstab(adata.obs['age_groups'], excl).to_string())
print("\nPENK present:", "PENK" in adata.var_names)

In [ ]:
globals().keys()

### Rebuild the state in your notebook

In [ ]:
os.environ["R_HOME"] = "/home/flavio/miniforge3/envs/renv/lib/R"
os.environ["R_LIBS_USER"] = "/home/flavio/miniforge3/envs/renv/lib/R/library"

from libs.run_scfates import preprocess, fit_curve, plot_trajectory, test_association, _ensure_r_home

_ensure_r_home()

In [ ]:
import rpy2.situation as s
print(s.get_r_home())

In [ ]:
# adata = assemble(...)  # you already have this: 1106 cells
sel = adata[~adata.obs["cell.types"].astype(str).str.startswith("IN_excl")].copy()
sel.obs["cell.types"] = sel.obs["cell.types"].astype(str).astype("category")   # drop empty levels

IN = preprocess(sel, n_top_genes=5000, n_neighbors=25, layer="counts")         # 870 x 5000
fit_curve(IN, root_gene="PENK", nodes=30, n_map=100, n_jobs=15, seed=42)       # published recipe

In [ ]:
IN

Object looks complete: 35 obs columns = your original 30 plus the four scFates outputs (t, seg, edge, t_sd) and milestones. What they mean:

- t — pseudotime, averaged over the 100 mappings
- t_sd — its standard deviation across those mappings (1.16, 0.73, 1.58 here, on a ~95-unit axis, so tightly determined)
- seg — which branch of the principal graph the cell sits on
- edge — the pair of principal points it falls between (7|10, 25|1)
- milestones — nearest milestone node; all three show 12, which is the root node PENK selected

Your first three cells illustrate the internal-root problem concretely: 
1. two Chrom_C and one Aut_Neu_1 
  - all terminally differentiated types — sit at t = 8.76, 10.40, 6.79, i.e. essentially at the origin. 
  - That's the signature of rooting mid-curve: pseudotime measures distance from node 12 in both directions, so a differentiated population ends up at the start.


In [ ]:
print(IN.obs.shape)

IN.obs.head(3).T

Two checks worth running on your object:

In [ ]:
# print(pd.crosstab(IN.obs["cell.types"], IN.obs["seg"]))          # which types share each branch
print(IN.uns["graph"]["pp_seg"])                                  # from/to nodes per branch
print("root:", IN.uns["graph"]["root"], "tips:", IN.uns["graph"]["tips"])   # root among tips?

pd.crosstab(IN.obs["cell.types"], IN.obs["seg"])

That explains everything. PENK marks the chromaffin end, not the progenitor end:

| type	| n	| PENK mean	|  % expressing | 
|-------|---|-----------|---------------|
| Chrom_C	| 41	| 0.88	| 65.9 | 
| Aut_Neu_1	| 145	| 0.13	| 15.9 | 
| SCP_1	| 168	| 0.02	| 2.4 | 
| SCP_5	| 195	| 0.00	| 2.6 | 
| Bridge_state	| 34	| 0.00	| 0.0 | 


scf.tl.root(adata, "PENK") picks the node with highest mean PENK — so it roots the trajectory in the chromaffin population, which sits between the two tips. Hence node 12 internal, tips [0, 2], and two branches radiating out:


1. seg 1:  12 → 0   d = 96.9   holds all 553 SCPs (plus neurons lying between root and SCP tip)
2. seg 2:  12 → 2   d = 34.5   holds the neuronal tip
3. 96.9 + 34.5 = 131.5, the full graph length — and my SCP-tip-rooted fit spans 129.8, i.e. the same single axis traversed end to end. 
- Rooting at tip 0 merges these two branches into one monotone segment.

Two consequences visible in your crosstab that matter for interpretation:

1. Pseudotime runs backwards on segment
  SCPs sit at high t there (up to 96.9) while chromaffin and neurons sit near t = 0. So on the larger branch, increasing pseudotime means progenitor-ward.
2. The neuronal types are split across both branches — Aut_Neu_1 49/96, Aut_Neu_2 29/40, Chrom_C 23/18. Any per-segment trend analysis on this rooting compares the same cell type against itself in two opposite directions.

So the published recipe is reproducible but its root choice is a marker-selection artifact: PENK is a terminal marker being used as a progenitor marker. Rooting at the SCP tip is the defensible version, and it's what the saved fit and all the module analysis use.

That closes the last open question in the analysis. Ready to assemble the methods note — version deviations, the root-choice reasoning per compartment, and links to every figure and table — unless you want anything else checked first.

### Then the tip-rooted variant, using the helper rather than a hard-coded node

- SCP ~ Schwann Cell Precursors, which are nerve-associated embryonic progenitor cells that display broad multipotent potential.

In [ ]:
import scFates as scf


def progenitor_tip(adata, progenitor_prefix="SCP", label_key="cell.types", n_cells=30):
    """Return the tip node whose closest cells are richest in `progenitor_prefix`."""
    R = np.asarray(adata.obsm["X_R"])            # soft assignment, cells x nodes
    labs = adata.obs[label_key].astype(str).values
    out = {}
    for tip in map(int, adata.uns["graph"]["tips"]):
        near = np.argsort(R[:, tip])[::-1][:n_cells]
        out[tip] = float(np.mean([l.startswith(progenitor_prefix) for l in labs[near]]))
    return max(out, key=out.get), out

IN_tip = IN.copy()
del IN_tip.uns["graph"]["root"]
tip, frac = progenitor_tip(IN)                   # {0: 0.27, 2: 0.0} -> tip 0
scf.tl.root(IN_tip, tip)
scf.tl.pseudotime(IN_tip, n_jobs=10, n_map=100, seed=42)

In [ ]:
# assert {'IN', 'IN_tip', 'pca', 'ct', 'order', 'pal'} <= globals().keys()

### The plotting variables were just locals in my figure cell:

In [ ]:
pca   = IN.obsm["X_pca"][:, :2]
ct    = IN.obs["cell.types"].astype(str)
order = ["SCP_1","SCP_2","SCP_3","SCP_4","SCP_5","My_SC","Bridge_state","Chrom_C","Aut_Neu_1","Aut_Neu_2"]
pal   = dict(zip(order, plt.get_cmap("tab10")(np.linspace(0, 1, 10))))

### Important

Two things worth knowing when you run it. 

- scf.tl.root refuses to overwrite an existing root, hence the del 
- and if you skip the .copy() you'll overwrite the PENK fit and lose the comparison. 

- And that 27% is not a commanding majority: 
  - the cells nearest tip 0 are a mix of SCPs and Bridge_state, which is the same signal that makes me think a tree is the better model here.

Everything above is also saved in innervation_pseudotime.csv and innervation_selection_scfates.h5ad, so you can skip the refit entirely — sc.read_h5ad on that object gives you IN with both pseudotimes already in obs.

In [ ]:
summ = pd.DataFrame({
    "n": IN.obs["cell.types"].value_counts(),
    "t_PENK": IN.obs.groupby("cell.types", observed=True)["t"].mean(),
    "t_sd_PENK": IN.obs.groupby("cell.types", observed=True)["t_sd"].median(),
    "t_SCPtip": IN_tip.obs.groupby("cell.types", observed=True)["t"].mean(),
    "t_sd_SCPtip": IN_tip.obs.groupby("cell.types", observed=True)["t_sd"].median(),
}).sort_values("t_SCPtip")

summ

In [ ]:
import warnings

warnings.simplefilter("ignore")

root_claude = Path("/home/flavio/.claude-science")
root_artifact = Path("orgs/abb023bb-bd59-418e-bf97-0ebfba525efa/artifacts/proj_c74c86fe51af/71854aa8-1956-4fe8-ace3-00fae28123e6/")
# moved to root_h5
fname_h5 = "vfd40245c_innervation_selection_scfates.h5ad"
fname = "innervation_celltype_markers.tsv"

filename = root_results / fname

if not filename.exists():
    IN = sc.read_h5ad(root_h5 / fname_h5)

    sc.tl.rank_genes_groups(IN, "cell.types", method="wilcoxon", use_raw=True, n_genes=10)
    mk = pd.DataFrame({g: [IN.uns["rank_genes_groups"]["names"][g][i] for i in range(8)]
                    for g in IN.obs["cell.types"].cat.categories})
    tab = pd.DataFrame({
        "n": IN.obs["cell.types"].value_counts(),
        "t_SCPtip": IN.obs.groupby("cell.types", observed=True)["t_SCPtip_root"].mean().round(1),
        "top_markers": {c: ", ".join(mk[c][:6]) for c in mk.columns},
    }).sort_values("t_SCPtip")


    tab.reset_index(inplace=True)
    cols = list(tab.columns)
    cols[0] = 'cell_type'
    tab.columns = cols

    tab.to_csv(root_results / fname, sep='\t', index=False)
else:
    tab = pd.read_csv(filename, sep='\t')

print(tab.shape)
tab

#### chromaffin transition 

Chromaffin transition primarily refers to the specialized cellular process where nerve-associated progenitor cells differentiate into neuroendocrine chromaffin cells within the adrenal medulla.

[Developmental heterogeneity of embryonic neuroendocrine chromaffin cells and their maturation dynamics - 2022](https://www.nature.com/articles/s41467-022-30438-w)  
[Serotonin limits generation of chromaffin cells during adrenal organ development - 2022](https://www.frontiersin.org/journals/endocrinology/articles/10.3389/fendo.2022.1020000/full)  


#### The Developmental Transition (Progenitor to Neuroendocrine)

For decades, textbooks taught that chromaffin cells derived directly from migrating neural crest cells. 

Modern single-cell transcriptomics has updated this model, revealing a precise transitional pathway:

1. Schwann Cell Precursors (SCPs): Progenitor cells migrate along preganglionic sympathetic nerves to reach the embryonic adrenal gland.
2. The "Bridge" State: Upon arriving at the adrenal primordium, SCPs transition into a short-lived, transient cell population known as "bridge" cells (frequently marked by the expression of the Htr3a gene).
3. Mature Chromaffin Cells: The bridge cells rapidly complete their transition into fully specialized chromaffin cells. Driven by core transcription factors like Ascl1 and Phox2b, they stop expressing glial markers, lose their ability to grow long axons, and instead acquire a secretory phenotype capable of packaging catecholamines into large dense-core granules.

#### SCP_1–5 — Schwann cell precursors (553 cells, 64% of the compartment). 
Neural crest–derived glial precursors that migrate along developing nerves and remain multipotent: they generate Schwann cells, and via the bridge state chromaffin cells. The five subsets are states rather than a maturation series. SCP_5 and SCP_1 carry matrix programs (COL1A1/COL1A2, SPARC, POSTN), with GFRA3 in SCP_5 a genuine neural-crest/SCP receptor. SCP_4 is the most glial-committed — PLP1 is the canonical Schwann-lineage myelin proteolipid, alongside NRXN3. SCP_2 carries SOX5 and PTPRZ1, both glial-progenitor associated.

#### SCP_3 ???
Deserves scepticism. Its entire top-ranked signature is heat-shock and stress response — HSP90AB1, HSP90AA1, HSPA1B, JUND. That's a dissociation-stress phenotype, not a cell identity, and its 38 cells probably shouldn't be read as a biological state at all. I'd exclude it from any pseudotime interpretation.

#### My_SC — myelinating Schwann cells (28). 
MBP (myelin basic protein), PMP22, S100B: the terminal glial fate. Only 28 cells because myelination barely starts before PCW 14.

#### Bridge_state (34). 
The SCP → chromaffin transition, described by Furlan et al. in mouse adrenal medulla. ASCL1 is the proneural transcription factor that drives sympathoadrenal specification, with TBX2 and TLX2 alongside it. This cluster is the mechanistic link underpinning the paper's chromaffin claim.

#### Chrom_C — chromaffin cells (41). 
CHGB (chromogranin B, the secretory-granule protein that defines chromaffin identity) plus DLK1 and NDUFA4L2. The abstract presents these as the first spatial account of chromaffin cells in the fetal human heart.

#### Aut_Neu_1 (145) — differentiated autonomic neurons. 
SYT1 (synaptotagmin-1, synaptic vesicle), ELAVL4 (HuD, neuron-specific RNA-binding), CADPS (dense-core vesicle exocytosis) — a synaptic-machinery program.

#### Aut_Neu_2 (69) — neurons in active axon outgrowth. 
Almost purely cytoskeletal: TUBB2B, TUBB3, TUBA1A, STMN2. That's a growth-cone/axonogenesis signature, consistent with its position at the far end of the pseudotime axis.

One inconsistency to keep in view: Bridge_state should be an intermediate between SCPs and chromaffin cells, but our tip-rooted curve places it at the lowest pseudotime of all ten — below every SCP. Combined with the finding that no tree configuration separates the chromaffin branch, this is the clearest sign that 870 cells (41 of them chromaffin) can't resolve the sympathoadrenal transition. The ordering is sound for the glia → neuron axis; the bridge → chromaffin step is not something this data supports.

In [ ]:
f"there are {summ.n.sum()} cells in the dataset"

In [ ]:
# relative uncertainty: sd as a fraction of the pseudotime range
rng = float(IN_tip.obs["t"].max() - IN_tip.obs["t"].min())
summ["t_sd_pct_range"] = (100 * summ["t_sd_SCPtip"] / rng)

fname = "innervation_tsd_by_celltype.tsv"
filename = root_results / fname

if not filename.exists():
    summ.round(2).to_csv(filename, sep="\t")

print(f"\ncurve range (SCP-tip root): {rng:.1f} | median t_sd {IN_tip.obs.t_sd.median():.2f} "
      f"({100*IN_tip.obs.t_sd.median()/rng:.1f}% of range) | 95th pct {IN_tip.obs.t_sd.quantile(.95):.2f}")
print("cells with t_sd > 5% of range:", int((IN_tip.obs.t_sd > 0.05*rng).sum()), "of", IN_tip.n_obs)

In [ ]:
summ

In [ ]:
summ.index

### Rendering uncertainty figure

In [ ]:
from libs.matplotlib_figure_style import *

### the uncertainty is spatially structured — plus a label collision and wasted y-range. Correcting:

In [ ]:
tsd = IN_tip.obs["t_sd"].values
tt = IN_tip.obs["t"].values

In [ ]:
IN_tip.obs["t_sd"]

In [ ]:
IN_tip.obs["t"]

In [ ]:
from libs.matplotlib_figure_style import *

In [ ]:
plt.close("all")
apply_figure_style(sizes=(8, 7, 6))

fig, axes = plt.subplots(1, 3, figsize=(12, 3),
                         gridspec_kw=dict(wspace=0.45, width_ratios=[1, 1, 1.15]))

ax = axes[0]
s = ax.scatter(pca[:, 0], pca[:, 1], s=7, c=tsd, cmap="magma_r", lw=0, rasterized=True)
ax.set_title("Projection uncertainty\nacross the embedding", fontsize=8, loc="left")
ax.set_xticks([]); ax.set_yticks([]); ax.margins(0.06)
for sp in ax.spines.values(): sp.set_visible(False)
cax = ax.inset_axes([0.0, -0.10, 0.5, 0.05])
cb = fig.colorbar(s, cax=cax, orientation="horizontal")
cb.set_label("t s.d.", fontsize=6, labelpad=1); cb.ax.tick_params(labelsize=6, pad=1)
ax.annotate("", xy=(0.15, 0.10), xytext=(0.02, 0.10), xycoords="axes fraction",
            arrowprops=dict(arrowstyle="->", lw=0.8, color="0.35"))
ax.annotate("", xy=(0.02, 0.23), xytext=(0.02, 0.10), xycoords="axes fraction",
            arrowprops=dict(arrowstyle="->", lw=0.8, color="0.35"))
ax.text(0.165, 0.095, "PC1", transform=ax.transAxes, fontsize=6, color="0.35", va="center")
ax.text(0.015, 0.245, "PC2", transform=ax.transAxes, fontsize=6, color="0.35")

ax = axes[1]
for t in order:
    m = (ct == t).values
    ax.scatter(tt[m], tsd[m], s=5, c=[pal[t]], lw=0, rasterized=True)
ax.set_xlabel("pseudotime (SCP-tip root)")
ax.set_ylabel("t s.d. over 100 mappings")
ax.set_title("Uncertainty stays below 3\non a 130-unit curve", fontsize=8, loc="left")
ax.margins(0.04); set_frame(ax)

ax = axes[2]
rs = np.random.default_rng(0)
for i, t in enumerate(summ.index):
    v = IN_tip.obs.loc[(ct == t).values, "t_sd"].values
    ax.scatter(v, i + rs.uniform(-0.22, 0.22, v.size), s=4, c=[pal[t]], lw=0, alpha=0.75)
    ax.plot([np.median(v)] * 2, [i - 0.34, i + 0.34], color="0.15", lw=1.4, solid_capstyle="butt")
ax.set_yticks(range(len(summ.index))); ax.set_yticklabels(summ.index, fontsize=6)
ax.invert_yaxis(); ax.set_xlabel("t s.d. over 100 mappings")
ax.set_title("Rows ordered by pseudotime;\nbar = median", fontsize=8, loc="left")
ax.margins(x=0.06, y=0.03); set_frame(ax)

for a, L in zip(axes, "abc"): panel_letter(a, L)
fig.savefig(root_figure / "pseudotime_uncertainty.png", dpi=300, bbox_inches="tight")
r = fig.canvas.get_renderer()
texts = [(t, t.get_window_extent(r)) for t in fig.findobj(mpl.text.Text) if t.get_text().strip() and t.get_visible()]
print("overlaps:", [(a.get_text()[:20], b.get_text()[:20]) for i,(a,ba) in enumerate(texts)
                    for b,bb in texts[i+1:] if ba.overlaps(bb)])
print("max t_sd:", round(float(tsd.max()), 2), "| curve length:", round(rng, 1))

### Projection uncertainty is negligible

median t_sd = 0.95 on a 129.8-unit curve — 0.7% of the range — with a 95th percentile of 2.32 and a maximum of 2.90. Not one of the 870 cells exceeds 5% of curve length. Per-type medians run from 0.51 (Aut_Neu_2) to 1.32 (SCP_2), so no cell type is systematically ill-placed. The values are identical under both roots, as expected — rooting shifts the origin, it doesn't change how consistently cells map to the graph.

#### The useful conclusion is a negative one

n_map=100 isn't buying much here. The ordering is tightly determined given the curve. What's actually uncertain is the curve itself — whether one line is the right topology — and no amount of mapping replication addresses that. n_map=1 would have given you the same picture in a fraction of the time, which is worth knowing before you run this on the larger compartments.

#### `Panel a` does show some spatial structure 

The higher values sit in the dense SCP region and at the neuronal end, where cells are equidistant from several nodes — but at a scale of ~2 units on a 130-unit axis it has no interpretive consequence.

- innervation_tsd_by_celltype.csv — per-type n, mean pseudotime under both roots, median t_sd, and t_sd as a percentage of curve length

- cardiomyocytes02.rds, endothelial.rds and fibroblasts.rds are downloaded and ready for step 6, but I need your call on the question from the last turn before fitting them, since it determines how each compartment gets rooted: keep the published curve + marker-root recipe, switch to tip-rooting, or refit as branching trees. The compartments are larger and genuinely branched (fibroblast heterogeneity is a stated focus of the paper), so the topology choice matters more there than it did here.

### Envirnoment vars needed: R_HOME, LD_LIBRARY_PATH

In [ ]:
import time, warnings, contextlib, io
warnings.simplefilter("ignore")

# bash
# export R_HOME=/home/flavio/miniforge3/envs/renv/lib/R
# export LD_LIBRARY_PATH=/home/flavio/miniforge3/envs/renv/lib/R/lib

# R
# install.packages('mgcv')
# Rscript -e 'library(mgcv); sessionInfo()'

# python -m rpy2.situation


os.environ["R_HOME"] = "/home/flavio/miniforge3/envs/renv/lib/R"
os.environ["R_LIBS_USER"] = "/home/flavio/miniforge3/envs/renv/lib/R/library"
# os.environ.pop('LD_LIBRARY_PATH', None)

import rpy2.robjects as ro 

In [ ]:
# !python -m rpy2.situation

In [ ]:
print("rpy2 ok | R:", ro.r("R.version.string")[0], "| mgcv:", ro.r('as.character(packageVersion("mgcv"))')[0])

In [ ]:
import rpy2
import importlib
print(importlib.import_module("scFates.tools.fit").rmgcv)

for m in [k for k in list(sys.modules) if k.split(".")[0] in ("scFates", "rpy2")]:
    del sys.modules[m]

import scFates as scf                                     # re-runs importeR with rpy2 working
print(importlib.import_module("scFates.tools.fit").rmgcv)  # expect: InstalledSTPackage ...
# scf.__dict__


That warning — R was initialized outside of rpy2 (R_NilValue != NULL). Trying to use it nevertheless. — is the failure mode I flagged, and it's now the problem. The old rpy2 had already initialised embedded R in that process; the fresh rpy2 attached to it "nevertheless", and the result is inconsistent: fit.rmgcv is a live Package while test_association's own flags are still error strings. Those are two separate importeR() calls (tools/__init__.py imports test_association before fit, and the first one ran against the half-working R), so they disagree.

There's no clean in-process recovery from two rpy2 instances over one embedded R. Restart the kernel — this is the case where it's mandatory.

After restart, make this the first cell, with nothing above it:

In [ ]:
sys.path.insert(0, "/home/flavio/uv/neural_crest/src")
print(os.environ.get("R_HOME"))        # must be non-None from kernel.json
import importlib, numpy as np, scanpy as sc, scFates as scf
for m in ("scFates.tools.test_association", "scFates.tools.fit"):
    print(m, "->", importlib.import_module(m).rmgcv)    # BOTH must be Package objects

In [ ]:
import os, sys, shutil, importlib
os.environ["PATH"] = "/home/flavio/miniforge3/envs/renv/bin" + os.pathsep + os.environ["PATH"]
print(shutil.which("R"))            # /home/flavio/miniforge3/envs/renv/bin/R

for k in [k for k in list(sys.modules) if k.split(".")[0] == "scFates"]:
    del sys.modules[k]
import scFates as scf

m = importlib.import_module("scFates.tools.test_association")
print(all(not isinstance(getattr(m, n), str) for n in ("Rpy2", "R", "rstats", "rmgcv", "Formula")))

In [ ]:
print(*os.environ["PATH"].split(os.pathsep)[:6], sep="\n")

In [ ]:
IN = sc.read_h5ad(root_h5 / "vfd40245c_innervation_selection_scfates.h5ad")
for c in ("seg", "milestones", "edge", "cell.types"):
    IN.obs[c] = IN.obs[c].astype(str).astype("category")

# reroot at the SCP tip (the saved graph is PENK-rooted)
R = np.asarray(IN.obsm["X_R"]); labs = IN.obs["cell.types"].astype(str).values
frac = {int(tp): float(np.mean([l.startswith("SCP") for l in labs[np.argsort(R[:, tp])[::-1][:30]]]))
        for tp in map(int, IN.uns["graph"]["tips"])}
del IN.uns["graph"]["root"]
scf.tl.root(IN, max(frac, key=frac.get))
scf.tl.pseudotime(IN, n_jobs=1, n_map=100, seed=42)

scf.tl.test_association(IN, n_jobs=15, fdr_cut=1e-4, A_cut=0.3)   # ~16 s
scf.tl.fit(IN, n_jobs=15)

In [ ]:

from libs.trends_figure import trends_figure
fig, mod = trends_figure(IN, "Innervation", root_figure / "trends_innervation.png", k=6)

### It is Ok

1,725 genes, module sizes 248/279/300/75/632/191, same representative genes per module. 

Yours is smoother because the tip-rooted fit has one segment, so pooled and per-segment binning coincide.

### M4 is the one to look at critically

M4: MYL7, TNNT2, TNNC1 are cardiomyocyte sarcomere genes, in a neural crest compartment. Finding where they come from:

In [ ]:
mods = pd.read_csv(host.artifact_path("03461c2e-4257-4b96-99b6-2d6c7a1a5a9c"), index_col=0)
ad = sc.read_h5ad(host.artifact_path("fd40245c-489b-4179-a5a2-f53c68a49e1e")) if False else \
     sc.read_h5ad(host.artifact_path(IDS["innervation_selection_scfates.h5ad"]))
m4 = [g for g in mods.index[mods["module"] == 4] if g in ad.raw.var_names]
X = np.asarray(ad.raw[:, m4].X.todense())
score = pd.Series(X.mean(1), index=ad.obs_names)

by_type = pd.DataFrame({
    "n": ad.obs["cell.types"].value_counts(),
    "M4_mean": score.groupby(ad.obs["cell.types"].values, observed=True).mean(),
    "pct_cells_top_decile": 100 * (score > score.quantile(0.9)).groupby(
        ad.obs["cell.types"].values, observed=True).mean(),
}).sort_values("M4_mean", ascending=False).round(3)
print(f"M4 = {len(m4)} genes; top: {', '.join(mods.index[mods['module']==4][:6])}\n")
print(by_type.to_string())
top_samples = (score.groupby(ad.obs["sampleID"].values).mean()
                 .sort_values(ascending=False).head(4).round(3))
print("\nhighest M4 by sample:", top_samples.to_dict())
print("cells above the 90th pct of M4:", int((score > score.quantile(0.9)).sum()),
      "| their median nCount:", float(ad.obs.loc[score > score.quantile(0.9), "nCount_RNA"].median()),
      "vs overall:", float(ad.obs["nCount_RNA"].median()))

In [ ]:
print(host.skills.publish("neural-crest-trajectory"))